# 04 - Validação de YAMLs

Notebook para validar YAMLs individualmente ou em lote.

**Útil para:**
- Re-validar após edições manuais
- Verificar status de todos os YAMLs
- Identificar padrões de erros

## 1. Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
SYSTEM_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = SYSTEM_DIR / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import paths
from validators import validate_yaml, validate_yaml_file, YAMLValidator

from qa_guideline import run_qa
from requote_from_texts import requote_failed, fix_review, enforce_file_pages_all

QA_THRESHOLD = 80

print('Modules loaded')


✅ Módulos carregados


## 2. Listar YAMLs Disponíveis

In [2]:
yamls_approved = sorted(paths.YAMLS.glob("*.yaml"))
yamls_work = sorted(paths.WORK.glob("*/extraction.yaml"))

print(f" YAMLs aprovados: {len(yamls_approved)}")
print(f" YAMLs em work/: {len(yamls_work)}")

# Mostrar lista
if yamls_approved:
    print("\nAprovados:")
    for y in yamls_approved[:10]:
        print(f"   {y.name}")
    if len(yamls_approved) > 10:
        print(f"   ... e mais {len(yamls_approved) - 10}")

📁 YAMLs aprovados: 38
📁 YAMLs em work/: 25

Aprovados:
   ART_001.yaml
   ART_002.yaml
   ART_003.yaml
   ART_004.yaml
   ART_005.yaml
   ART_006.yaml
   ART_007.yaml
   ART_008.yaml
   ART_009.yaml
   ART_010.yaml
   ... e mais 28


## 3. Validar YAML Individual

In [ ]:
# Selecionar artigo para validar
ARTIGO_ID = "ART_001"

yaml_path = paths.YAMLS / f"{ARTIGO_ID}.yaml"

if yaml_path.exists():
    print(f" Validando: {yaml_path}")
    result = validate_yaml_file(yaml_path)
    print(result)
else:
    # Tentar em work/
    yaml_path = paths.WORK / ARTIGO_ID / "extraction.yaml"
    if yaml_path.exists():
        print(f" Validando: {yaml_path}")
        result = validate_yaml_file(yaml_path)
        print(result)
    else:
        print(f" YAML não encontrado para {ARTIGO_ID}")

## 4. Validar Todos os YAMLs

In [3]:
results = []

# Validar todos os YAMLs aprovados
for yaml_path in sorted(paths.YAMLS.glob("*.yaml")):
    result = validate_yaml_file(yaml_path)
    results.append({
        "ArtigoID": yaml_path.stem,
        "Status": "" if result.is_valid else "",
        "Erros": len(result.errors),
        "Avisos": len(result.warnings),
        "Regras_OK": len(result.rules_passed),
        "Regras_Falha": result.rules_failed
    })

# Criar DataFrame
df = pd.DataFrame(results)

if not df.empty:
    print(f" Resumo: {len(df)} YAMLs validados")
    approved = len(df[df['Status'] == ''])
    failed = len(df[df['Status'] == ''])
    print(f"    Aprovados: {approved}")
    print(f"    Com erros: {failed}")
    
    df
else:
    print("Nenhum YAML para validar")

📊 Resumo: 38 YAMLs validados
   ✅ Aprovados: 38
   ❌ Com erros: 0


## 4B. QA do Guia v3.3 (pós-validação)

- **Default threshold:** 80 (permite pequena flexibilidade vs. literal)
- Status: **OK / REVIEW / FAIL** (FAIL = precisa correção antes de usar no mestrado)


In [ ]:
# 1) Rodar QA e exportar relatório (Guia v3.3)
qa_df, qa_report_path = run_qa(threshold=QA_THRESHOLD, export=True)
print(f"QA report: {qa_report_path}")
display(qa_df)

# Resumo rápido
status_counts = qa_df['status'].value_counts()
print(status_counts.to_string())

# 0) (Opcional) Normalizar Página para sempre ser a página do arquivo local (p.<n>)
# Útil quando o PDF é um recorte de volume (ex.: viewer mostra 506 (1/30)).
# Descomente para aplicar em todos os YAMLs e depois rode o QA novamente.
# pages_log_df, pages_log_path = enforce_file_pages_all(threshold=QA_THRESHOLD)
# print(f"Enforce file pages log: {pages_log_path}")
# display(pages_log_df)

# 2) Auto-corrigir FAILs (só se existir FAIL)
# Descomente para corrigir e depois rode o QA novamente.
# if status_counts.get('FAIL', 0) > 0:
#     requote_log_df, requote_log_path = requote_failed(qa_df, threshold=QA_THRESHOLD)
#     print(f"Requote FAIL log: {requote_log_path}")
#     display(requote_log_df)
# else:
#     print('Sem FAIL — pule requote_failed.')

# 3) Auto-corrigir REVIEWs (preencher páginas p.NR e/ou requotar quando match_rate < 0.80)
# Descomente para tentar elevar REVIEW -> OK automaticamente.
# if status_counts.get('REVIEW', 0) > 0:
#     review_fix_df, review_fix_path = fix_review(qa_df, threshold=QA_THRESHOLD)
#     print(f"Review fix log: {review_fix_path}")
#     display(review_fix_df)
# else:
#     print('Sem REVIEW — nada a corrigir.')


## 5. Detalhes de Erros

In [4]:
# Mostrar detalhes dos YAMLs com erros
print(" Detalhes dos erros:\n")

for yaml_path in sorted(paths.YAMLS.glob("*.yaml")):
    result = validate_yaml_file(yaml_path)
    
    if not result.is_valid:
        print(f" {yaml_path.stem}:")
        for e in result.errors:
            print(f"   • {e}")
        print()

🔍 Detalhes dos erros:



## 6. Análise de Padrões de Erros

In [5]:
# Contar erros por tipo/regra
error_counts = {}
rule_failures = {}

for yaml_path in sorted(paths.YAMLS.glob("*.yaml")):
    result = validate_yaml_file(yaml_path)
    
    for e in result.errors:
        # Extrair tipo de erro (ex: [R1], [SCHEMA], etc.)
        if e.startswith("["):
            error_type = e.split("]")[0] + "]"
        else:
            error_type = "[OUTRO]"
        error_counts[error_type] = error_counts.get(error_type, 0) + 1
    
    for r in result.rules_failed:
        rule_failures[f"R{r}"] = rule_failures.get(f"R{r}", 0) + 1

print(" Erros por tipo:")
for error_type, count in sorted(error_counts.items(), key=lambda x: -x[1]):
    print(f"   {error_type}: {count}")

print("\n Regras mais violadas:")
for rule, count in sorted(rule_failures.items(), key=lambda x: -x[1]):
    print(f"   {rule}: {count} violações")

📊 Erros por tipo:

📊 Regras mais violadas:


## 7. Validar YAML de Texto (Cole aqui)

In [ ]:
# Cole um YAML aqui para validar
yaml_text = """
# Cole seu YAML aqui
"""

if yaml_text.strip() and yaml_text.strip() != "# Cole seu YAML aqui":
    result = validate_yaml(yaml_text)
    print(result)
else:
    print("Cole um YAML na variável yaml_text para validar")

## 8. Exportar Relatório de Validação

In [6]:
# Exportar para CSV
if not df.empty:
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    report_path = paths.CONSOLIDATED / f"validation_report_{timestamp}.csv"
    
    df.to_csv(report_path, index=False)
    print(f" Relatório exportado: {report_path}")
else:
    print("Nenhum dado para exportar")

✅ Relatório exportado: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\Extraction\outputs\consolidated\validation_report_20260203_1117.csv
